In [30]:
%reload_ext autoreload
%autoreload 2

import numpy as np

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

from fpp.models.scd import dnds
from fpp.likelihoods.npll_jax import log_like_np
from fpp.likelihoods.npll_jax_ import log_like_np as log_like_np_log
from fpp.models.np_model import NPModel

import numpyro
from numpyro import handlers
from einops import repeat

import json

In [17]:
model = NPModel()
vd = json.load(open("../outputs/truths/truth_dict_base230927.json", "r"))

Number of pixels in ROI: 6839
Loading the psf correction from: /n/home07/yitians/fermi/fermi-prob-prog/cross_check/psf_dir/Fermi_PSF_2GeV2_nside128.npy
Max photon count is 103


In [31]:
def model_old(data=model.data):
    
    # Get mixed pib template
    theta_pib = np.array(vd["theta_pib"])
    temp_pib = jnp.sum(theta_pib[:, None] * model.pib, 0)

    # Get mixed ics template
    theta_ics = np.array(vd["theta_ics"])
    temp_ics = jnp.sum(theta_ics[:, None] * model.ics, 0)

    S_gce = vd["S_gce"]
        
    temps = [model.temp_iso, model.temp_bub, model.temp_psc, temp_pib, temp_ics]
    temp_labels = ["iso", "bub", "psc", "pib", "ics"]
            
    mu = jnp.zeros_like(data)
    
    for temp, temp_label in zip(temps, temp_labels):
        
        if temp_label in ["pib", "ics"]:
            prior_lo, prior_hi = 1e-3, 14.
        else:
            prior_lo, prior_hi = 1e-3, 5.0
            
        S_temp = vd["S_{}".format(temp_label)]
        
        if temp_label in ["pib"]:
            
            temp_pib_mod = jnp.zeros_like(data)
            for ii in range(len(model.Ylm_temps)):
                Alm = vd["Alm_{}".format(ii)]
                temp_pib_mod += Alm * model.Ylm_temps[ii]
            
            temp_pib_mod = (1. + temp_pib_mod) * temp
            
            A_temp = S_temp / jnp.mean(temp_pib_mod[~model.nm])
            mu += A_temp * temp_pib_mod  
        else:
            A_temp = S_temp / jnp.mean(temp[~model.nm])
            mu += A_temp * temp     
                                        
    gamma_ps = vd["gamma_ps"]
    gamma_poiss = vd["gamma_poiss"]

    temp_gce_nfw_ps = model.nfw_temp_gen.get_NFW2_template(gamma=gamma_ps)
    temp_gce_nfw_poiss = model.nfw_temp_gen.get_NFW2_template(gamma=gamma_poiss)
        
    zs = vd["zs"]
    C = vd["C"]
    temp_dsk = model.dsk_temp_gen.get_template(zs=zs, C=C)
            
    f_bulge_ps = vd["f_bulge_ps"]
    f_bulge_poiss = vd["f_bulge_poiss"]
    
    theta_bulge_poiss = np.array(vd["theta_bulge_poiss"])
    temp_bulge = jnp.sum(theta_bulge_poiss[:, None] * model.blg_s, 0)
    
    # Normalize to same mean
    A_gce_nfw = S_gce / jnp.mean(temp_gce_nfw_poiss[~model.nm])
    A_gce_bulge = S_gce / jnp.mean(temp_bulge[~model.nm])
    temp_gce_poiss = (1 - f_bulge_poiss) * A_gce_nfw * temp_gce_nfw_poiss \
                        + f_bulge_poiss * A_gce_bulge * temp_bulge
    
    A_gce = S_gce / jnp.mean(temp_gce_poiss[~model.nm])
    mu += A_gce * temp_gce_poiss
    
    # Get mixed bulge template
    theta_bulge_ps = np.array(vd["theta_bulge_ps"])
    temp_bulge = jnp.sum(theta_bulge_ps[:, None] * model.blg_s, 0)

    # Normalize to same mean
    A_gce_nfw = 1 / jnp.mean(temp_gce_nfw_ps[~model.nm])
    A_gce_bulge = 1 / jnp.mean(temp_bulge[~model.nm])

    # Get hybrid template
    temp_gce_ps = (1 - f_bulge_ps) * A_gce_nfw * temp_gce_nfw_ps + f_bulge_ps * A_gce_bulge * temp_bulge

    npt_compressed = jnp.array([temp_gce_ps, temp_dsk])

    theta = []    

    for ips, ps in enumerate(["gce", "dsk"]):

        Sps = vd["Sps_{}".format(ps)]

        n1 = vd["n1_{}".format(ps)]
        n2 = vd["n2_{}".format(ps)]
        n3 = vd["n3_{}".format(ps)]
        sb1 = vd["sb1_{}".format(ps)]
        lambda_s = vd["lambdas_{}".format(ps)]

        theta_tmp = jnp.array([1., n1, n2, n3, sb1, lambda_s * sb1])

        s_ary = jnp.logspace(-1., 2., 1000)
        dnds_ary = dnds(s_ary, theta_tmp)

        A = Sps / jnp.mean(npt_compressed[ips][~model.nm] * jnp.trapz(s_ary * dnds_ary, s_ary))

        theta.append([A, n1, n2, n3, sb1, lambda_s * sb1])

    theta = jnp.array(theta)
            
    # Pad the last exposure region so that all are the same size
    exp_lens = [len(model.expreg_indices[i]) for i in range(len(model.expreg_indices))]
    n_pad = exp_lens[0] - exp_lens[-1]
    
    expreg_indices = jnp.zeros_like(model.expreg_indices)
    expreg_indices = expreg_indices.at[:-1].set(model.expreg_indices[:-1])
    expreg_indices = expreg_indices.at[-1].set(jnp.pad(model.expreg_indices[-1], (0, n_pad)))

    log_like_np_exp_vmapped = jax.vmap(log_like_np, in_axes=(0, 0, 1, 0, None, None, None, None))
            
    # Get relevant arrays for different exposure regions
    mu_batch = mu[~model.mask_roi][jnp.array(expreg_indices)]
    npt_compressed_batch = npt_compressed[:, ~model.mask_roi][:, jnp.array(expreg_indices)]
    data_batch = data[~model.mask_roi][jnp.array(expreg_indices)]
    
    exposure_multiplier = model.exposure_means_list / model.exposure_mean
    
    # Scale non-Poissonian parameters (norm divided by exposure ratio, breaks multiplied)
    theta = repeat(theta, "n_ps n_param -> n_exp n_ps n_param", n_exp=len(expreg_indices))
    theta = theta.at[:, :, 0].set(theta[:, :, 0] / exposure_multiplier[:, None])
    theta = theta.at[:, :, -1].set(theta[:, :, -1] * exposure_multiplier[:, None])
    theta = theta.at[:, :, -2].set(theta[:, :, -2] * exposure_multiplier[:, None])
    
    with numpyro.plate("data", size=len(mu[~model.mask_roi]), dim=-1):
        
        log_like_exp = log_like_np_exp_vmapped(theta, mu_batch, npt_compressed_batch, data_batch, model.f_ary, model.df_rho_ary, model.k_max, len(expreg_indices[0]))
        
        # Concatenate exposure regions
        loglike = jnp.concatenate(log_like_exp)[:len(mu[~model.mask_roi])]
                            
        with handlers.mask(mask=~jnp.logical_or(jnp.isinf(loglike), jnp.isnan(loglike))):
            return loglike


def model_new(data=model.data):
        """Main numpyro model."""

        mu = jnp.zeros_like(data)

        #=== pib with spherical harmonics, ics ===
        theta_pib = np.array(vd["theta_pib"])
        temp_pib = jnp.sum(theta_pib[:, None] * model.pib, 0)
        theta_ics = np.array(vd["theta_ics"])
        temp_ics = jnp.sum(theta_ics[:, None] * model.ics, 0)

        pib_modifier = jnp.zeros_like(data)
        for i in range(len(model.Ylm_temps)):
            Alm = vd[f'Alm_{i}']
            pib_modifier += Alm * model.Ylm_temps[i]
        temp_pib = (1 + pib_modifier) * temp_pib
        temp_pib /= jnp.mean(temp_pib[~model.nm]) # re-normalize after modulation

        mu += vd["S_pib"] * temp_pib
        mu += vd["S_ics"] * temp_ics

        #=== other fixed diffuse templates ===
        mu += vd["S_iso"] * model.temp_iso
        mu += vd["S_bub"] * model.temp_bub
        mu += vd["S_psc"] * model.temp_psc

        #=== diffuse gce (blg + nfw) ===
        S_gce = vd["S_gce"]
        f_bulge_poiss = vd["f_bulge_poiss"]

        theta_blg_poiss = np.array(vd["theta_bulge_poiss"])
        temp_blg_poiss = jnp.sum(theta_blg_poiss[:, None] * model.blg_s, 0)

        temp_nfw_poiss = model.nfw_temp_gen.get_NFW2_template(gamma=vd["gamma_poiss"])
        temp_nfw_poiss /= jnp.mean(temp_nfw_poiss[~model.nm])

        mu += S_gce * (f_bulge_poiss * temp_blg_poiss + (1 - f_bulge_poiss) * temp_nfw_poiss)
                                            
        #=== PS gce (blg + nfw) ===
        Sps_gce = vd["Sps_gce"]
        f_bulge_ps = vd["f_bulge_ps"]

        theta_blg_ps = np.array(vd["theta_bulge_ps"])
        temp_blg_ps = jnp.sum(theta_blg_ps[:, None] * model.blg_s, 0)

        temp_nfw_ps = model.nfw_temp_gen.get_NFW2_template(gamma=vd["gamma_ps"])
        temp_nfw_ps /= jnp.mean(temp_nfw_ps[~model.nm])

        temp_gce_ps = f_bulge_ps * temp_blg_ps + (1 - f_bulge_ps) * temp_nfw_ps

        #=== PS disk ===
        Sps_dsk = vd["Sps_dsk"]
        zs = vd["zs"]
        C = vd["C"]
        temp_dsk_ps = model.dsk_temp_gen.get_template(zs=zs, C=C)
        temp_dsk_ps /= jnp.mean(temp_dsk_ps[~model.nm])
        
        #=== PS processing ===
        Sps_list = [Sps_gce, Sps_dsk]
        npt_compressed = jnp.array([temp_gce_ps, temp_dsk_ps])
        theta = []
        s_arr = jnp.logspace(-1., 2., 1000)
        for i, ps in enumerate(["gce", "dsk"]):
            n1 = vd[f'n1_{ps}']
            n2 = vd[f'n2_{ps}']
            n3 = vd[f'n3_{ps}']
            sb1 = vd[f'sb1_{ps}']
            lambda_s = vd[f'lambdas_{ps}']

            theta_tmp = jnp.array([1., n1, n2, n3, sb1, lambda_s * sb1])
            dnds_arr = dnds(s_arr, theta_tmp)
            A = Sps_list[i] / jnp.trapz(s_arr * dnds_arr, s_arr)
            theta.append([A, n1, n2, n3, sb1, lambda_s * sb1])
        theta = jnp.array(theta)
        
        #=== exposure regions ===
        # pad the last exposure region so that all are the same size
        exp_lens = [len(model.expreg_indices[i]) for i in range(len(model.expreg_indices))]
        n_pad = exp_lens[0] - exp_lens[-1]
        
        expreg_indices = jnp.zeros_like(model.expreg_indices)
        expreg_indices = expreg_indices.at[:-1].set(model.expreg_indices[:-1])
        expreg_indices = expreg_indices.at[-1].set(jnp.pad(model.expreg_indices[-1], (0, n_pad)))

        log_like_np_exp_vmapped = jax.vmap(log_like_np_log, in_axes=(0, 0, 1, 0, None, None, None, None))
                
        mu_batch = mu[~model.mask_roi][jnp.array(expreg_indices)]
        npt_compressed_batch = npt_compressed[:, ~model.mask_roi][:, jnp.array(expreg_indices)]
        data_batch = data[~model.mask_roi][jnp.array(expreg_indices)]
        exposure_multiplier = model.exposure_means_list / model.exposure_mean
        
        # scale non-Poissonian parameters (norm divided by exposure ratio, breaks multiplied)
        theta = repeat(theta, "n_ps n_param -> n_exp n_ps n_param", n_exp=len(expreg_indices))
        theta = theta.at[:, :, 0].set(theta[:, :, 0] / exposure_multiplier[:, None])
        theta = theta.at[:, :, -1].set(theta[:, :, -1] * exposure_multiplier[:, None])
        theta = theta.at[:, :, -2].set(theta[:, :, -2] * exposure_multiplier[:, None])

        #=== likelihood ===
        with numpyro.plate("data", size=len(mu[~model.mask_roi]), dim=-1):
            
            log_like_exp = log_like_np_exp_vmapped(
                theta,
                mu_batch,
                npt_compressed_batch,
                data_batch,
                model.f_ary,
                model.df_rho_ary,
                model.k_max,
                len(expreg_indices[0])
            )
            loglike = jnp.concatenate(log_like_exp)[:len(mu[~model.mask_roi])]

            with handlers.mask(mask=~jnp.logical_or(jnp.isinf(loglike), jnp.isnan(loglike))):
                return loglike

In [22]:
model_old()

Array([-2.75284248, -4.36856119, -3.78662064, ..., -3.02157223,
       -2.4305252 , -3.00016361], dtype=float64)

In [28]:
model_new()

Array([-2.75284248, -4.36856119, -3.78662064, ..., -3.02157223,
       -2.4305252 , -3.00016361], dtype=float64)

In [32]:
model_old()/model_new()

Array([1., 1., 1., ..., 1., 1., 1.], dtype=float64)